<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Segmentation — Implementation</b></h1>
</div>

## Setup — Environment and Configuration

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from scipy import ndimage

import cv2

np.set_printoptions(precision=3, suppress=True)

print("NumPy :", np.__version__)
print("OpenCV:", cv2.__version__)
print("Setup : PASS")

### 0.1 Locate the Lab Automatically


In [ ]:
def locate_lab_root():
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks" / "main.ipynb").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the segmentation lab root."
    )
LAB_ROOT = locate_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

assert DATA_FILES, f"No supported images found in: {DATA_DIR}"
def select_input_image(role: str) -> Path:
    role = role.lower()

    exact = [
        path for path in DATA_FILES
        if path.stem.lower() == role
    ]
    if exact:
        return exact[0]

    matches = [
        path for path in DATA_FILES
        if role in path.stem.lower()
    ]
    if not matches:
        raise FileNotFoundError(
            f"No image matching role '{role}' found in {DATA_DIR}"
        )
    return matches[0]
INPUT_IMAGES = {
    role: select_input_image(role)
    for role in ("hand", "tower", "peppers")
}

print("Lab root         :", LAB_ROOT)
print("Data dir         :", DATA_DIR)
print("Images discovered:", len(DATA_FILES))
print("Experiment roles :", len(INPUT_IMAGES))
print("Output           :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
def load_grayscale_image(path):
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.uint8
    )
def load_rgb_image(path):
    return np.asarray(
        Image.open(path).convert("RGB"),
        dtype=np.uint8
    )
def display_grayscale_image(ax, image, title):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
def display_rgb_image(ax, image, title):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
def save_output_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)
def convert_mask_to_uint8(mask):
    return (
        np.asarray(mask).astype(bool) * 255
    ).astype(np.uint8)
def overlay_segmentation_mask(image_rgb, mask, alpha=0.35):

    output = image_rgb.astype(np.float32).copy()

    mask_bool = np.asarray(mask).astype(bool)

    red = np.zeros_like(output)
    red[..., 0] = 255

    output[mask_bool] = (
        (1 - alpha) * output[mask_bool]
        + alpha * red[mask_bool]
    )

    return np.clip(output, 0, 255).astype(np.uint8)

## 1. Segmentation Problem Formulation

In [ ]:
def create_threshold_mask(
    image,
    threshold,
    foreground="dark",
):
    image = np.asarray(image)
    if image.ndim != 2:
        raise ValueError("create_threshold_mask expects a 2-D grayscale image.")

    if foreground == "dark":
        return image < threshold

    if foreground == "bright":
        return image >= threshold

    raise ValueError("foreground must be 'dark' or 'bright'.")
FOREGROUND = True
BACKGROUND = False

print(
    "Mask convention:",
    {"foreground": FOREGROUND, "background": BACKGROUND},
)

## 2. Load the Lab Images


In [ ]:
hand_image = load_grayscale_image(INPUT_IMAGES["hand"])
tower_image_rgb = load_rgb_image(INPUT_IMAGES["tower"])
peppers_image_rgb = load_rgb_image(INPUT_IMAGES["peppers"])

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    hand_image,
    "Hand"
)

display_rgb_image(
    axes[1],
    tower_image_rgb,
    "Tower"
)

display_rgb_image(
    axes[2],
    peppers_image_rgb,
    "Peppers"
)

fig.tight_layout()
save_output_figure(
    fig,
    "01_input_images.png"
)
plt.show()

print("hand shape   :", hand_image.shape)
print("tower shape  :", tower_image_rgb.shape)
print("peppers shape:", peppers_image_rgb.shape)

## 3. Histogram-Based Threshold Selection


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

display_grayscale_image(
    axes[0],
    hand_image,
    "Hand image"
)
axes[1].hist(
    hand_image.ravel(),
    bins=256,
    range=(0, 255)
)
axes[1].set_title(
    "Hand intensity histogram"
)
axes[1].set_xlabel(
    "Intensity"
)
axes[1].set_ylabel(
    "Pixel count"
)

fig.tight_layout()
save_output_figure(
    fig,
    "02_hand_histogram.png"
)
plt.show()

## 4. Manual Global Thresholding


In [ ]:
manual_threshold = 120
manual_threshold_mask = create_threshold_mask(hand_image, manual_threshold, foreground="dark")

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    hand_image,
    "Original"
)

display_grayscale_image(
    axes[1],
    manual_threshold_mask,
    f"Mask — T={manual_threshold}"
)
segmented_hand_image = np.where(
    manual_threshold_mask,
    hand_image,
    255
)

display_grayscale_image(
    axes[2],
    segmented_hand_image,
    "Segmented foreground"
)

fig.tight_layout()
save_output_figure(
    fig,
    "03_manual_threshold.png"
)
plt.show()

## 5. Threshold Sensitivity


In [ ]:
threshold_values = [
    70,
    100,
    130,
    160
]

fig, axes = plt.subplots(
    1,
    len(threshold_values),
    figsize=(16, 4)
)

for ax, threshold in zip(
    axes,
    threshold_values
):

    mask = hand_image < threshold

    display_grayscale_image(
        ax,
        mask,
        f"T={threshold}"
    )

fig.tight_layout()
save_output_figure(
    fig,
    "04_threshold_sensitivity.png"
)
plt.show()

## 6. Otsu Thresholding


In [ ]:
otsu_threshold, mask_otsu_cv = cv2.threshold(
    hand_image,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)
otsu_mask = (
    mask_otsu_cv > 0
)

print(
    "Otsu threshold:",
    otsu_threshold
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    hand_image,
    "Original"
)

display_grayscale_image(
    axes[1],
    otsu_mask,
    f"Otsu mask — T={otsu_threshold:.1f}"
)

segmentation_overlay = overlay_segmentation_mask(
    np.stack([hand_image] * 3, axis=-1),
    otsu_mask
)

display_rgb_image(
    axes[2],
    segmentation_overlay,
    "Mask overlay"
)

fig.tight_layout()
save_output_figure(
    fig,
    "05_otsu_threshold.png"
)
plt.show()

## 7. Gaussian Smoothing Before Thresholding


In [ ]:
smoothed_hand_image = cv2.GaussianBlur(

    hand_image,
    (5, 5),
    0
)

blur_otsu_threshold, blur_otsu_cv = cv2.threshold(
    smoothed_hand_image,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

smoothed_otsu_mask = (
    blur_otsu_cv > 0
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    hand_image,
    "Original"
)

display_grayscale_image(
    axes[1],
    smoothed_hand_image,
    "Gaussian-smoothed"
)

display_grayscale_image(
    axes[2],
    smoothed_otsu_mask,
    "Otsu after smoothing"
)

fig.tight_layout()
save_output_figure(
    fig,
    "06_smoothing_before_otsu.png"
)
plt.show()

## 8. Adaptive Thresholding


In [ ]:
adaptive_mean_mask = cv2.adaptiveThreshold(
    hand_image,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

adaptive_gaussian_mask = cv2.adaptiveThreshold(
    hand_image,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    hand_image,
    "Original"
)

display_grayscale_image(
    axes[1],
    adaptive_mean_mask,
    "Adaptive mean"
)

display_grayscale_image(
    axes[2],
    adaptive_gaussian_mask,
    "Adaptive Gaussian"
)

fig.tight_layout()
save_output_figure(
    fig,
    "07_adaptive_thresholding.png"
)
plt.show()

## 9. Morphological Processing

In [ ]:
def apply_morphological_operation(
    mask,
    operation,
    kernel,
    iterations=1,
):
    mask_u8 = convert_mask_to_uint8(mask)
    if iterations < 1:
        raise ValueError("iterations must be at least 1.")

    if operation == "erode":
        result = cv2.erode(mask_u8, kernel, iterations=iterations)
    elif operation == "dilate":
        result = cv2.dilate(mask_u8, kernel, iterations=iterations)
    elif operation == "open":
        result = cv2.morphologyEx(
            mask_u8, cv2.MORPH_OPEN, kernel, iterations=iterations
        )
    elif operation == "close":
        result = cv2.morphologyEx(
            mask_u8, cv2.MORPH_CLOSE, kernel, iterations=iterations
        )
    else:
        raise ValueError(
            "operation must be 'erode', 'dilate', 'open', or 'close'."
        )

    return result > 0
print("Supported morphology:", ["erode", "dilate", "open", "close"])

## 10. Structuring Elements


In [ ]:
kernel_rect = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (7, 7)
)
kernel_ellipse = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)
kernel_cross = cv2.getStructuringElement(
    cv2.MORPH_CROSS,
    (7, 7)
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

display_grayscale_image(
    axes[0],
    kernel_rect,
    "Rectangle"
)

display_grayscale_image(
    axes[1],
    kernel_ellipse,
    "Ellipse"
)

display_grayscale_image(
    axes[2],
    kernel_cross,
    "Cross"
)

fig.tight_layout()
save_output_figure(
    fig,
    "08_structuring_elements.png"
)
plt.show()

## 11. Erosion and Dilation


In [ ]:
binary_mask_uint8 = convert_mask_to_uint8(
    smoothed_otsu_mask
)
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

eroded_mask = cv2.erode(
    binary_mask_uint8,
    kernel,
    iterations=1
)

dilated_mask = cv2.dilate(
    binary_mask_uint8,
    kernel,
    iterations=1
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    binary_mask_uint8,
    "Original mask"
)

display_grayscale_image(
    axes[1],
    eroded_mask,
    "Erosion"
)

display_grayscale_image(
    axes[2],
    dilated_mask,
    "Dilation"
)

fig.tight_layout()
save_output_figure(
    fig,
    "09_erosion_dilation.png"
)
plt.show()

## 12. Opening and Closing


In [ ]:
opened_mask = cv2.morphologyEx(
    binary_mask_uint8,
    cv2.MORPH_OPEN,
    kernel
)

closed_mask = cv2.morphologyEx(
    binary_mask_uint8,
    cv2.MORPH_CLOSE,
    kernel
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    binary_mask_uint8,
    "Original mask"
)

display_grayscale_image(
    axes[1],
    opened_mask,
    "Opening"
)

display_grayscale_image(
    axes[2],
    closed_mask,
    "Closing"
)

fig.tight_layout()
save_output_figure(
    fig,
    "10_opening_closing.png"
)
plt.show()

## 13. Morphological Gradient


In [ ]:
morphological_gradient = cv2.morphologyEx(
    binary_mask_uint8,
    cv2.MORPH_GRADIENT,
    kernel
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

display_grayscale_image(
    axes[0],
    binary_mask_uint8,
    "Mask"
)

display_grayscale_image(
    axes[1],
    morphological_gradient,
    "Morphological gradient"
)

fig.tight_layout()
save_output_figure(
    fig,
    "11_morphological_gradient.png"
)
plt.show()

## 14. Hole Filling


In [ ]:
binary_mask_with_holes = smoothed_otsu_mask.astype(bool)

hole_filled_mask = ndimage.binary_fill_holes(
    binary_mask_with_holes
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_grayscale_image(
    axes[0],
    binary_mask_with_holes,
    "Before filling"
)

display_grayscale_image(
    axes[1],
    hole_filled_mask,
    "After filling"
)

display_grayscale_image(
    axes[2],
    hole_filled_mask.astype(int)
    - binary_mask_with_holes.astype(int),
    "Pixels added"
)

fig.tight_layout()
save_output_figure(
    fig,
    "12_hole_filling.png"
)
plt.show()

## 15. Connected Components


In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    convert_mask_to_uint8(hole_filled_mask),
    connectivity=8
)

print(
    "Number of foreground components:",
    num_labels - 1
)
component_areas = stats[
    1:,
    cv2.CC_STAT_AREA
]

print(
    "Foreground areas:",
    component_areas
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

display_grayscale_image(
    axes[0],
    hole_filled_mask,
    "Binary mask"
)

axes[1].imshow(
    labels,
    cmap="nipy_spectral"
)
axes[1].set_title(
    "Connected-component labels"
)
axes[1].axis("off")

fig.tight_layout()
save_output_figure(
    fig,
    "13_connected_components.png"
)
plt.show()

## 16. Remove Small Components


In [ ]:
minimum_area = 500
area_filtered_components = np.zeros_like(
    labels,
    dtype=bool
)

for label_id in range(
    1,
    num_labels
):
    area = stats[
        label_id,
        cv2.CC_STAT_AREA
    ]
    if area >= minimum_area:
        area_filtered_components |= (
            labels == label_id
        )

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

display_grayscale_image(
    axes[0],
    hole_filled_mask,
    "Before area filtering"
)

display_grayscale_image(
    axes[1],
    area_filtered_components,
    f"Area ≥ {minimum_area}"
)

fig.tight_layout()
save_output_figure(
    fig,
    "14_component_area_filtering.png"
)
plt.show()

## 17. Contours


In [ ]:
contours, hierarchy = cv2.findContours(
    convert_mask_to_uint8(
        area_filtered_components
    ),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

hand_rgb = np.stack(
    [hand_image] * 3,
    axis=-1
)
contour_overlay = hand_rgb.copy()

cv2.drawContours(
    contour_overlay,
    contours,
    -1,
    (255, 0, 0),
    2
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

display_grayscale_image(
    axes[0],
    area_filtered_components,
    "Clean mask"
)

display_rgb_image(
    axes[1],
    contour_overlay,
    "Detected contours"
)

fig.tight_layout()
save_output_figure(
    fig,
    "15_contours.png"
)
plt.show()

print(
    "Number of external contours:",
    len(contours)
)

## 18. Region Properties


In [ ]:
region_measurements = []

for index, contour in enumerate(
    contours,
    start=1
):

    area = cv2.contourArea(
        contour
    )

    perimeter = cv2.arcLength(
        contour,
        True
    )

    x, y, w, h = cv2.boundingRect(
        contour
    )

    moments = cv2.moments(
        contour
    )
    if moments["m00"] != 0:
        cx = moments["m10"] / moments["m00"]
        cy = moments["m01"] / moments["m00"]
    else:
        cx = np.nan
        cy = np.nan

    circularity = (
        4 * np.pi * area
        / (perimeter ** 2)

        if perimeter > 0
        else np.nan
    )
    aspect_ratio = (
        w / h

        if h > 0
        else np.nan
    )

    region_measurements.append(
        {
            "component": index,
            "area": area,
            "perimeter": perimeter,
            "cx": cx,
            "cy": cy,
            "width": w,
            "height": h,
            "aspect_ratio": aspect_ratio,
            "circularity": circularity,
        }
    )

for row in region_measurements:
    print(row)

## 19. Color Segmentation


In [ ]:
peppers_image_hsv = cv2.cvtColor(
    peppers_image_rgb,
    cv2.COLOR_RGB2HSV
)

hue = peppers_image_hsv[..., 0]

saturation = peppers_image_hsv[..., 1]
value = peppers_image_hsv[..., 2]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(16, 4)
)

display_rgb_image(
    axes[0],
    peppers_image_rgb,
    "RGB"
)

display_grayscale_image(
    axes[1],
    hue,
    "Hue"
)

display_grayscale_image(
    axes[2],
    saturation,
    "Saturation"
)

display_grayscale_image(
    axes[3],
    value,
    "Value"
)

fig.tight_layout()
save_output_figure(
    fig,
    "16_hsv_channels.png"
)
plt.show()

### HSV Range Experiment


In [ ]:
lower_red_1 = np.array(
    [0, 80, 50],
    dtype=np.uint8
)

upper_red_1 = np.array(
    [12, 255, 255],
    dtype=np.uint8
)

lower_red_2 = np.array(
    [165, 80, 50],
    dtype=np.uint8
)

upper_red_2 = np.array(
    [179, 255, 255],
    dtype=np.uint8
)
mask_red_1 = cv2.inRange(
    peppers_image_hsv,
    lower_red_1,
    upper_red_1
)
mask_red_2 = cv2.inRange(
    peppers_image_hsv,
    lower_red_2,
    upper_red_2
)
red_region_mask = (
    (mask_red_1 > 0)
    | (mask_red_2 > 0)
)

segmented_red_regions = peppers_image_rgb.copy()
segmented_red_regions[~red_region_mask] = 0

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_rgb_image(
    axes[0],
    peppers_image_rgb,
    "Original peppers"
)

display_grayscale_image(
    axes[1],
    red_region_mask,
    "Red-color mask"
)

display_rgb_image(
    axes[2],
    segmented_red_regions,
    "Segmented red regions"
)

fig.tight_layout()
save_output_figure(
    fig,
    "17_color_segmentation.png"
)
plt.show()

## 20. Edge-Based Segmentation


In [ ]:
tower_image_gray = cv2.cvtColor(
    tower_image_rgb,
    cv2.COLOR_RGB2GRAY
)
smoothed_tower_image = cv2.GaussianBlur(
    tower_image_gray,
    (5, 5),
    0
)
tower_edge_map = cv2.Canny(
    smoothed_tower_image,
    80,
    160
)
edge_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (5, 5)
)
closed_tower_edge_map = cv2.morphologyEx(
    tower_edge_map,
    cv2.MORPH_CLOSE,
    edge_kernel,
    iterations=2
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

display_rgb_image(
    axes[0],
    tower_image_rgb,
    "Tower"
)

display_grayscale_image(
    axes[1],
    tower_edge_map,
    "Canny edges"
)

display_grayscale_image(
    axes[2],
    closed_tower_edge_map,
    "Closed edge map"
)

fig.tight_layout()
save_output_figure(
    fig,
    "18_edge_based_segmentation.png"
)
plt.show()

## 21. Distance Transform


In [ ]:
distance_transform_map = cv2.distanceTransform(
    convert_mask_to_uint8(area_filtered_components),
    cv2.DIST_L2,
    5
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

display_grayscale_image(
    axes[0],
    area_filtered_components,
    "Binary mask"
)

im = axes[1].imshow(
    distance_transform_map,
    cmap="viridis"
)

axes[1].set_title(
    "Distance transform"
)

axes[1].axis("off")

fig.colorbar(
    im,
    ax=axes[1],
    fraction=0.046
)

fig.tight_layout()
save_output_figure(
    fig,
    "19_distance_transform.png"
)
plt.show()

## 22. Watershed Segmentation


In [ ]:
watershed_source_image = peppers_image_rgb.copy()

peppers_image_gray = cv2.cvtColor(
    peppers_image_rgb,
    cv2.COLOR_RGB2GRAY
)

_, peppers_binary = cv2.threshold(
    peppers_image_gray,
    0,
    255,
    cv2.THRESH_BINARY
    + cv2.THRESH_OTSU
)
ws_kernel = np.ones(
    (3, 3),
    np.uint8
)

opening = cv2.morphologyEx(
    peppers_binary,
    cv2.MORPH_OPEN,
    ws_kernel,
    iterations=2
)
sure_background = cv2.dilate(
    opening,
    ws_kernel,

iterations=3
)
distance_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)
_, sure_foreground = cv2.threshold(
    distance_transform,
    0.5 * distance_transform.max(),
    255,
    0
)

sure_foreground = np.uint8(
    sure_foreground
)
unknown_region = cv2.subtract(
    sure_background,
    sure_foreground
)

num_markers, markers = cv2.connectedComponents(
    sure_foreground
)
markers = markers + 1
markers[unknown_region == 255] = 0

watershed_result_labels = cv2.watershed(
    cv2.cvtColor(
        watershed_source_image,
        cv2.COLOR_RGB2BGR
    ),
    markers.copy()
)
watershed_boundary_overlay = watershed_source_image.copy()
watershed_boundary_overlay[
    watershed_result_labels == -1
] = [255, 0, 0]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

display_rgb_image(
    axes[0, 0],
    peppers_image_rgb,
    "Input"
)

display_grayscale_image(
    axes[0, 1],
    opening,
    "Opening"
)

display_grayscale_image(
    axes[0, 2],
    sure_background,
    "Sure background"
)

display_grayscale_image(
    axes[1, 0],
    distance_transform,
    "Distance transform"
)

display_grayscale_image(
    axes[1, 1],
    sure_foreground,
    "Sure foreground"
)

display_rgb_image(
    axes[1, 2],
    watershed_boundary_overlay,
    "Watershed boundaries"
)

fig.tight_layout()
save_output_figure(
    fig,
    "20_watershed.png"
)
plt.show()

## 23. Ground Truth and Segmentation Metrics


In [ ]:
def compute_segmentation_metrics(
    ground_truth,
    prediction
):
    ground_truth_mask = np.asarray(ground_truth).astype(bool)
    predicted_mask = np.asarray(prediction).astype(bool)
    true_positives = np.logical_and(ground_truth_mask, predicted_mask).sum()
    true_negatives = np.logical_and(~ground_truth_mask, ~predicted_mask).sum()
    false_positives = np.logical_and(~ground_truth_mask, predicted_mask).sum()
    false_negatives = np.logical_and(ground_truth_mask, ~predicted_mask).sum()
    epsilon = 1e-12
    accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives + epsilon)
    precision = true_positives / (true_positives + false_positives + epsilon)
    recall = true_positives / (true_positives + false_negatives + epsilon)
    iou = true_positives / (true_positives + false_positives + false_negatives + epsilon)
    dice = 2 * true_positives / (2 * true_positives + false_positives + false_negatives + epsilon)

    return {
        "TP": int(true_positives),
        "TN": int(true_negatives),
        "FP": int(false_positives),
        "FN": int(false_negatives),
        "Accuracy": float(accuracy),
        "Precision": float(precision),
        "Recall": float(recall),
        "IoU": float(iou),
        "Dice": float(dice),
    }

### Synthetic Ground-Truth Check


In [ ]:
demo_gt = area_filtered_components.copy()

demo_prediction = cv2.erode(
    convert_mask_to_uint8(
        area_filtered_components
    ),
    np.ones(
        (7, 7),
        np.uint8
    ),
    iterations=1
) > 0

demo_metrics = compute_segmentation_metrics(
    demo_gt,
    demo_prediction
)

for key, value in demo_metrics.items():

    if isinstance(value, float):
        print(
            f"{key:10s}: {value:.4f}"
        )
    else:
        print(
            f"{key:10s}: {value}"
        )

## 24. Dice vs IoU Relationship

In [ ]:
demo_iou = demo_metrics["IoU"]
demo_dice = demo_metrics["Dice"]
dice_from_iou = 2.0 * demo_iou / (1.0 + demo_iou)
iou_from_dice = demo_dice / (2.0 - demo_dice)

assert np.isclose(dice_from_iou, demo_dice)
assert np.isclose(iou_from_dice, demo_iou)

print(f"IoU           : {demo_iou:.4f}")
print(f"Dice          : {demo_dice:.4f}")
print(f"Dice from IoU : {dice_from_iou:.4f}")
print(f"IoU from Dice : {iou_from_dice:.4f}")

## 25. Under-Segmentation vs Over-Segmentation

In [ ]:
comparison_kernel = np.ones((9, 9), dtype=np.uint8)
under_segmented = cv2.erode(
    convert_mask_to_uint8(demo_gt),
    comparison_kernel,
    iterations=1,
) > 0
over_segmented = cv2.dilate(
    convert_mask_to_uint8(demo_gt),
    comparison_kernel,
    iterations=1,
) > 0

under_metrics = compute_segmentation_metrics(demo_gt, under_segmented)
over_metrics = compute_segmentation_metrics(demo_gt, over_segmented)

print(
    "Under-segmentation:",
    f"Precision={under_metrics['Precision']:.4f}",
    f"Recall={under_metrics['Recall']:.4f}",
    f"IoU={under_metrics['IoU']:.4f}",
)

print(
    "Over-segmentation:",
    f"Precision={over_metrics['Precision']:.4f}",
    f"Recall={over_metrics['Recall']:.4f}",
    f"IoU={over_metrics['IoU']:.4f}",
)

## 26. End-to-End Binary Segmentation Pipeline


In [ ]:
def segment_dark_foreground_object(
    image_gray,
    blur_kernel=(5, 5),
    minimum_area=500
):

    blurred = cv2.GaussianBlur(image_gray, blur_kernel, 0)

    threshold_value, binary = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5),
    )

    cleaned = apply_morphological_operation(
        binary > 0,
        operation="open",
        kernel=kernel,
        iterations=1,
    )

    cleaned = apply_morphological_operation(
        cleaned,
        operation="close",
        kernel=kernel,
        iterations=1,
    )

    hole_filled_mask = ndimage.binary_fill_holes(cleaned)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        convert_mask_to_uint8(hole_filled_mask),
        connectivity=8,
    )
    final_mask = np.zeros_like(hole_filled_mask, dtype=bool)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area >= minimum_area:
            final_mask |= labels == label_id

    return {
        "threshold": float(threshold_value),
        "blurred": blurred,
        "binary": binary > 0,
        "cleaned": cleaned,
        "filled": hole_filled_mask,
        "mask": final_mask,
    }

## 27. Segmentation Method Selection Criteria

In [ ]:
def recommend_segmentation_strategy(
    uneven_illumination=False,
    color_is_discriminative=False,
    touching_objects=False,
):

    if touching_objects:
        return "distance transform + marker-controlled watershed"
    if color_is_discriminative:
        return "color-space thresholding (for example HSV)"
    if uneven_illumination:
        return "adaptive thresholding"

    return "global thresholding / Otsu"
selection_examples = {
    "uniform grayscale object": recommend_segmentation_strategy(),
    "uneven lighting": recommend_segmentation_strategy(
        uneven_illumination=True
    ),
    "distinctive color": recommend_segmentation_strategy(
        color_is_discriminative=True
    ),
    "touching objects": recommend_segmentation_strategy(
        touching_objects=True
    ),
}

for case, method in selection_examples.items():
    print(f"{case:24s} -> {method}")

## 28. Integrated Segmentation Workflow

In [ ]:
hand_result = segment_dark_foreground_object(
    hand_image,
    minimum_area=500
)
final_hand_mask = hand_result[
    "mask"
]

final_overlay = overlay_segmentation_mask(
    np.stack(
        [hand_image] * 3,
        axis=-1
    ),
    final_hand_mask
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

display_grayscale_image(
    axes[0, 0],
    hand_image,
    "Input"
)

display_grayscale_image(
    axes[0, 1],
    hand_result["blurred"],
    "Smoothed"
)

display_grayscale_image(
    axes[0, 2],
    hand_result["binary"],
    "Otsu threshold"
)

display_grayscale_image(
    axes[1, 0],
    hand_result["cleaned"],
    "Morphological cleanup"
)

display_grayscale_image(
    axes[1, 1],
    final_hand_mask,
    "Final mask"
)

display_rgb_image(
    axes[1, 2],
    final_overlay,
    "Final overlay"
)

fig.tight_layout()
save_output_figure(
    fig,
    "21_complete_pipeline.png"
)
plt.show()

print(
    "Pipeline Otsu threshold:",
    hand_result["threshold"]
)
REQUIRED_OUTPUTS = [
    "01_input_images.png",
    "02_hand_histogram.png",
    "03_manual_threshold.png",
    "04_threshold_sensitivity.png",
    "05_otsu_threshold.png",
    "06_smoothing_before_otsu.png",
    "07_adaptive_thresholding.png",
    "08_structuring_elements.png",
    "09_erosion_dilation.png",
    "10_opening_closing.png",
    "11_morphological_gradient.png",
    "12_hole_filling.png",
    "13_connected_components.png",
    "14_component_area_filtering.png",
    "15_contours.png",
    "16_hsv_channels.png",
    "17_color_segmentation.png",
    "18_edge_based_segmentation.png",
    "19_distance_transform.png",
    "20_watershed.png",
    "21_complete_pipeline.png",
]
missing_outputs = [
    output_name
    for output_name in REQUIRED_OUTPUTS
    if not (OUTPUT_DIR / output_name).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: " + ", ".join(missing_outputs)
    )

print(f"Output validation passed: {len(REQUIRED_OUTPUTS)} files.")
